# Jointure **F5_2 ↔ F6_1** — Table bridge

## Problème

`F5_2_LCP_Energy_Emissions` et `F6_1_IED_Installations` n'ont **pas de clé commune** et n'ont **pas le même grain** → jointure directe impossible.

## Solution

Construire une **table bridge** qui relie `LCPInspireId` (F5) à `InstallationInspireId` + `parent_facilityInspireId` (F6) via quatre signaux :

| Signal | Colonnes utilisées | Poids |
|---|---|---|
| Distance géographique | `Latitude`, `Longitude` | 55 % |
| Similarité de nom | `installationPartName` ↔ `installationName` | 30 % |
| Même ville | `City_Of_Facility` ↔ `City_of_Facility` | 10 % |
| Pattern ID | `LCPInspireId` `.PART` → `.INSTALLATION` | 5 % |

## Ce que ce notebook produit

| Variable | Fichier CSV | Grain |
|---|---|---|
| `bridge_best` | `bridge_f5_f6_belgium.csv` | 1 ligne par `LCPInspireId` |
| `bridge_candidates` | `bridge_f5_f6_belgium_candidates.csv` | `LCPInspireId × candidat` |
| `f5_enriched` | `f5_enriched_with_bridge_belgium.csv` | `LCPInspireId × reportingYear × featureType` |

Chaque ligne du bridge porte un `match_score`, un `confidence_level` et un flag `manual_review_required`.

In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

from rapidfuzz.fuzz import token_set_ratio
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Paramètres du notebook

In [5]:
import os
print(os.getcwd())

/home/olivierpi/technofutur/15-final_project/belgian-environement-monitor/notebooks


In [6]:
TARGET_COUNTRY = "Belgium"
TOP_K = 5
MAX_DISTANCE_KM = 3.0

def find_file(filename: str) -> Path:
    candidates = [
    Path(".") / filename,
    Path("data/raw") / filename,
    Path("..") / "data" / "raw" / filename,
    Path("/mnt/") / filename,
    Path.cwd() / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Impossible de trouver {filename}")

F5_PATH = find_file("F5_2_LCP_Energy_Emissions.csv")
F6_PATH = find_file("F6_1_IED_Installations.csv")

print("F5_PATH =", F5_PATH)
print("F6_PATH =", F6_PATH)


F5_PATH = ../data/raw/F5_2_LCP_Energy_Emissions.csv
F6_PATH = ../data/raw/F6_1_IED_Installations.csv


## 2. Chargement minimal des colonnes utiles

In [7]:
f5_cols = [
    "countryName", "reportingYear", "LCPInspireId",
    "installationPartName", "City_Of_Facility",
    "Longitude", "Latitude", "featureType", "unit", "featureValue"
]

f6_cols = [
    "CountryName", "reportingYear", "parent_facilityInspireId",
    "InstallationInspireId", "installationName",
    "City_of_Facility", "Longitude", "Latitude", "installationStatus"
]

f5 = pd.read_csv(F5_PATH, usecols=f5_cols)
f6 = pd.read_csv(F6_PATH, usecols=f6_cols)

f5 = f5[f5["countryName"] == TARGET_COUNTRY].copy()
f6 = f6[f6["CountryName"] == TARGET_COUNTRY].copy()

print("F5 lignes :", len(f5))
print("F6 lignes :", len(f6))
print("F5 LCP uniques :", f5["LCPInspireId"].nunique())
print("F6 installations uniques :", f6["InstallationInspireId"].nunique())


F5 lignes : 10521
F6 lignes : 20129
F5 LCP uniques : 114
F6 installations uniques : 2702


## 3. Grain des deux tables

| Table | Grain | Conséquence |
|---|---|---|
| **F5_2** | `LCPInspireId × reportingYear × featureType` | Plusieurs lignes par installation par an |
| **F6_1** | `InstallationInspireId × reportingYear` | 1 ligne par installation par an |

Joindre directement les lignes brutes → **multiplication des lignes** et faux doublons.

**Bonne approche :**
1. `groupby("LCPInspireId")` → **1 ligne par LCP** (`build_lcp_representative`)
2. `groupby("InstallationInspireId")` → **1 ligne par installation** (`build_installation_representative`)
3. Construire le bridge sur ces représentants

In [8]:
display(
    f5[["LCPInspireId", "reportingYear", "featureType"]].head(5),
    f6[["InstallationInspireId", "reportingYear", "parent_facilityInspireId"]].head(5)
)


,LCPInspireId,reportingYear,featureType
10,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000069.PART,2018,LCPCharacteristics
26,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000060.PART,2016,LCPCharacteristics
28,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000067.PART,2016,LCPCharacteristics
45,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000054.PART,2021,LCPCharacteristics
54,BE.WA/095010101.PART,2020,LCPCharacteristics


,InstallationInspireId,reportingYear,parent_facilityInspireId
135,BE.WA/297010100.INSTALLATION,2023,BE.WA/297010000.FACILITY
136,BE.WA/099010100.INSTALLATION,2023,BE.WA/099010000.FACILITY
137,BE.WA/047010100.INSTALLATION,2023,BE.WA/047010000.FACILITY
138,BE.WA/240010100.INSTALLATION,2023,BE.WA/240010000.FACILITY
139,BE.WA/010010400.INSTALLATION,2023,BE.WA/010010000.FACILITY


### Lecture de l'aperçu

**F5** — grain : `LCPInspireId × reportingYear × featureType`

| Colonne | Rôle |
|---|---|
| `LCPInspireId` | Clé de la partie LCP — URL flamande ou code court `BE.WA/…` |
| `reportingYear` | Année de déclaration |
| `featureType` | Type de mesure — **crée plusieurs lignes par installation et par an** |

**F6** — grain : `InstallationInspireId × reportingYear`

| Colonne | Rôle |
|---|---|
| `InstallationInspireId` | Clé de l'installation IED — ex. `BE.WA/297010100.INSTALLATION` |
| `reportingYear` | Année de déclaration |
| `parent_facilityInspireId` | Site parent — ex. `BE.WA/297010000.FACILITY` |

---

### Comment le code évite la multiplication des lignes

| Étape | Opération clé | Grain résultant |
|---|---|---|
| Cellule 5 | `groupby("LCPInspireId").agg(...)` | 1 ligne par `LCPInspireId` |
| Cellule 5 | `groupby("InstallationInspireId").agg(...)` | 1 ligne par `InstallationInspireId` |
| Cellule 7 | `groupby("LCPInspireId").first()` | `bridge_best` : 1 ligne par `LCPInspireId` |
| Cellule 11 | `f5.merge(bridge_best, on="LCPInspireId", how="left")` | Many-to-one — grain de F5 préservé |

> ⚠️ `f5_enriched` garde le grain `LCPInspireId × reportingYear × featureType`. Pour joindre ensuite avec F6 sur `InstallationInspireId + reportingYear`, agréger d'abord au niveau `LCPInspireId × reportingYear` (voir cellule 12).

## 4. Fonctions utilitaires

In [9]:
def normalize_text(value):
    # Minuscules, suppression des accents, alphanumérique uniquement, espaces normalisés
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value or None

def candidate_from_lcp_id(lcp_id):
    # Heuristique : LCPInspireId se terminant par .PART → remplace par .INSTALLATION
    if pd.isna(lcp_id):
        return None
    s = str(lcp_id)
    if s.endswith(".PART"):
        return s[:-5] + ".INSTALLATION"
    return None

def prepare_balltree(df_points):
    # Index spatial haversine sur (Latitude, Longitude) converties en radians
    coords = np.deg2rad(df_points[["Latitude", "Longitude"]].to_numpy())
    return BallTree(coords, metric="haversine")


## 5. Construire une version représentative de F5_2 et F6_1

In [10]:
def build_lcp_representative(df_f5):
    tmp = df_f5.copy()
    tmp["name_norm"] = tmp["installationPartName"].map(normalize_text)
    tmp["city_norm"] = tmp["City_Of_Facility"].map(normalize_text)

    rep = (
        tmp.groupby("LCPInspireId")
        .agg(
            country=("countryName", "first"),
            installationPartName=("installationPartName", lambda s: s.dropna().iloc[0] if s.dropna().size else None),
            name_norm=("name_norm", lambda s: s.dropna().iloc[0] if s.dropna().size else None),
            City_Of_Facility=("City_Of_Facility", lambda s: s.dropna().mode().iloc[0] if s.dropna().size else None),
            city_norm=("city_norm", lambda s: s.dropna().mode().iloc[0] if s.dropna().size else None),
            Longitude=("Longitude", "median"),
            Latitude=("Latitude", "median"),
            min_reportingYear=("reportingYear", "min"),
            max_reportingYear=("reportingYear", "max"),
        )
        .reset_index()
    )
    return rep

def build_installation_representative(df_f6):
    tmp = df_f6.copy()
    tmp["name_norm"] = tmp["installationName"].map(normalize_text)
    tmp["city_norm"] = tmp["City_of_Facility"].map(normalize_text)

    rep = (
        tmp.groupby("InstallationInspireId")
        .agg(
            country=("CountryName", "first"),
            parent_facilityInspireId=("parent_facilityInspireId", lambda s: s.dropna().iloc[0] if s.dropna().size else None),
            installationName=("installationName", lambda s: s.dropna().iloc[0] if s.dropna().size else None),
            name_norm=("name_norm", lambda s: s.dropna().iloc[0] if s.dropna().size else None),
            City_of_Facility=("City_of_Facility", lambda s: s.dropna().mode().iloc[0] if s.dropna().size else None),
            city_norm=("city_norm", lambda s: s.dropna().mode().iloc[0] if s.dropna().size else None),
            Longitude=("Longitude", "median"),
            Latitude=("Latitude", "median"),
            min_reportingYear=("reportingYear", "min"),
            max_reportingYear=("reportingYear", "max"),
        )
        .reset_index()
    )
    return rep

lcp_rep = build_lcp_representative(f5).dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)
inst_rep = build_installation_representative(f6).dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

print("LCP représentatifs :", len(lcp_rep))
print("Installations représentatives :", len(inst_rep))

display(lcp_rep.head(3), inst_rep.head(3))


LCP représentatifs : 114
Installations représentatives : 2702


,LCPInspireId,country,installationPartName,name_norm,City_Of_Facility,city_norm,Longitude,Latitude,min_reportingYear,max_reportingYear
0,BE.BRU/100010002.PART,Belgium,TURBOJET BUDA,turbojet buda,Bruxelles,bruxelles,4.41120,50.906700,2016,2020
1,BE.BRU/100010003.PART,Belgium,TURBOJET VOLTA,turbojet volta,Bruxelles,bruxelles,4.39550,50.813900,2016,2023
2,BE.EEA/BE0015.PART,Belgium,Centrale Elec. Electrabel Awirs t5,centrale elec electrabel awirs t5,FLEMALLE-HAUTE,flemalle haute,5.42264,50.586396,2016,2016


,InstallationInspireId,country,parent_facilityInspireId,installationName,name_norm,City_of_Facility,city_norm,Longitude,Latitude,min_reportingYear,max_reportingYear
0,BE.BRU/100010001.INSTALLATION,Belgium,BE.BRU/100010001.FACILITY,ABATTOIR SA,abattoir sa,Anderlecht,anderlecht,4.32797,50.84363,2017,2024
1,BE.BRU/100010002.INSTALLATION,Belgium,BE.BRU/100010002.FACILITY,CORDEN PHARMA BRUSSELS,corden pharma brussels,Bruxelles,bruxelles,4.40365,50.90178,2017,2024
2,BE.BRU/100010003.INSTALLATION,Belgium,BE.BRU/100010003.FACILITY,SABCA,sabca,Bruxelles,bruxelles,4.41759,50.88191,2017,2024


## 6. Stratégie de matching

Pour chaque `LCPInspireId` :

1. **Filtrage** — priorité aux installations avec `city_norm` identique ; sinon pays entier
2. **Sélection** — `TOP_K` plus proches via `BallTree` sur (`Latitude`, `Longitude`)
3. **Scoring** :

| Signal | Colonnes → variable calculée | Poids |
|---|---|---|
| Distance géographique | `Latitude`, `Longitude` → `distance_km` | **55 %** |
| Similarité de nom | `name_norm` (LCP) ↔ `name_norm` (install.) → `name_similarity` | **30 %** |
| Même ville | `city_norm` → `same_city_flag` | **10 %** |
| Pattern ID | `LCPInspireId` `.PART → .INSTALLATION` → `id_guess_match` | **5 %** |

Les poids sont modifiables dans `compute_bridge_candidates`.

In [11]:
def compute_bridge_candidates(lcp_rep, inst_rep, top_k=5, max_distance_km=3.0):
    inst_all = inst_rep.copy().reset_index(drop=True)
    tree_all = prepare_balltree(inst_all)

    results = []

    for _, row in lcp_rep.iterrows():
        # Filtrage city_norm : candidats de la même ville → sinon tout le pays
        city_candidates = (
            inst_all[inst_all["city_norm"] == row["city_norm"]].reset_index(drop=True)
            if row["city_norm"] else pd.DataFrame()
        )

        if len(city_candidates) > 0:
            search_df = city_candidates
            search_tree = prepare_balltree(search_df)
            same_city_search_used = True
        else:
            search_df = inst_all
            search_tree = tree_all
            same_city_search_used = False

        k = min(top_k, len(search_df))
        distances, indices = search_tree.query(
            np.deg2rad([[row["Latitude"], row["Longitude"]]]),
            k=k
        )

        for rank, (dist_rad, idx) in enumerate(zip(distances[0], indices[0]), start=1):
            cand = search_df.iloc[int(idx)]
            distance_km = float(dist_rad * 6371.0088)

            name_similarity = float(
                token_set_ratio(str(row["name_norm"] or ""), str(cand["name_norm"] or ""))
            )
            same_city_flag = bool(row["city_norm"] and row["city_norm"] == cand["city_norm"])
            id_guess_match = bool(candidate_from_lcp_id(row["LCPInspireId"]) == cand["InstallationInspireId"])

            distance_score = max(0.0, 100.0 * (1 - min(distance_km, max_distance_km) / max_distance_km))
            city_score = 100.0 if same_city_flag else 0.0
            id_score = 100.0 if id_guess_match else 0.0

            match_score = (
                0.55 * distance_score +
                0.30 * name_similarity +
                0.10 * city_score +
                0.05 * id_score
            )

            results.append({
                "LCPInspireId": row["LCPInspireId"],
                "InstallationInspireId": cand["InstallationInspireId"],
                "parent_facilityInspireId": cand["parent_facilityInspireId"],
                "lcp_name": row["installationPartName"],
                "installation_name": cand["installationName"],
                "lcp_city": row["City_Of_Facility"],
                "installation_city": cand["City_of_Facility"],
                "lcp_longitude": row["Longitude"],
                "lcp_latitude": row["Latitude"],
                "installation_longitude": cand["Longitude"],
                "installation_latitude": cand["Latitude"],
                "distance_km": distance_km,
                "name_similarity": name_similarity,
                "same_city_flag": same_city_flag,
                "id_guess_match": id_guess_match,
                "same_city_search_used": same_city_search_used,
                "candidate_rank": rank,
                "match_score": round(match_score, 2),
            })

    return pd.DataFrame(results).sort_values(
        ["LCPInspireId", "match_score"], ascending=[True, False]
    )

bridge_candidates = compute_bridge_candidates(
    lcp_rep=lcp_rep,
    inst_rep=inst_rep,
    top_k=TOP_K,
    max_distance_km=MAX_DISTANCE_KM
)

print("Nombre de candidats générés :", len(bridge_candidates))
display(bridge_candidates.head(10))


Nombre de candidats générés : 537


,LCPInspireId,InstallationInspireId,parent_facilityInspireId,lcp_name,installation_name,lcp_city,installation_city,lcp_longitude,lcp_latitude,installation_longitude,installation_latitude,distance_km,name_similarity,same_city_flag,id_guess_match,same_city_search_used,candidate_rank,match_score
0,BE.BRU/100010002.PART,BE.BRU/100010011.INSTALLATION,BE.BRU/100010010.FACILITY,TURBOJET BUDA,TURBOJET VOLTA,Bruxelles,Bruxelles,4.4112,50.9067,4.41115,50.90672,0.004152,76.190476,True,False,True,1,87.78
1,BE.BRU/100010002.PART,BE.BRU/100010005.INSTALLATION,BE.BRU/100010005.FACILITY,TURBOJET BUDA,PRODAMTEX,Bruxelles,Bruxelles,4.4112,50.9067,4.40469,50.90981,0.572659,36.363636,True,False,True,2,65.41
2,BE.BRU/100010002.PART,BE.BRU/100010002.INSTALLATION,BE.BRU/100010002.FACILITY,TURBOJET BUDA,CORDEN PHARMA BRUSSELS,Bruxelles,Bruxelles,4.4112,50.9067,4.40365,50.90178,0.761302,28.571429,True,True,True,3,64.61
3,BE.BRU/100010002.PART,BE.BRU/100010008.INSTALLATION,BE.BRU/100010007.FACILITY,TURBOJET BUDA,CERES NV,Bruxelles,Bruxelles,4.4112,50.9067,4.40268,50.89583,1.348298,19.047619,True,False,True,4,46.00
4,BE.BRU/100010002.PART,BE.BRU/100010014.INSTALLATION,BE.BRU/100010014.FACILITY,TURBOJET BUDA,DERICHEBOURG SA,Bruxelles,Bruxelles,4.4112,50.9067,4.39632,50.88970,2.159232,28.571429,True,False,True,5,33.99
5,BE.BRU/100010003.PART,BE.BRU/100010012.INSTALLATION,BE.BRU/100010011.FACILITY,TURBOJET VOLTA,TURBOJET VOLTA,Bruxelles,Bruxelles,4.3955,50.8139,4.39546,50.81385,0.006230,100.000000,True,False,True,1,94.89
6,BE.BRU/100010003.PART,BE.BRU/100010009.INSTALLATION,BE.BRU/100010008.FACILITY,TURBOJET VOLTA,Forest Metals Vorst,Bruxelles,Bruxelles,4.3955,50.8139,4.31448,50.80576,5.764277,42.424242,True,False,True,2,22.73
8,BE.BRU/100010003.PART,BE.BRU/100010003.INSTALLATION,BE.BRU/100010003.FACILITY,TURBOJET VOLTA,SABCA,Bruxelles,Bruxelles,4.3955,50.8139,4.41759,50.88191,7.719762,21.052632,True,True,True,4,21.32
7,BE.BRU/100010003.PART,BE.BRU/100010010.INSTALLATION,BE.BRU/100010009.FACILITY,TURBOJET VOLTA,VIANGRO SA,Bruxelles,Bruxelles,4.3955,50.8139,4.30422,50.81443,6.413352,25.000000,True,False,True,3,17.50
9,BE.BRU/100010003.PART,BE.BRU/100010004.INSTALLATION,BE.BRU/100010004.FACILITY,TURBOJET VOLTA,BRUXELLES ENERGIE,Bruxelles,Bruxelles,4.3955,50.8139,4.38037,50.88310,7.767670,19.354839,True,False,True,5,15.81


## 7. Garder le meilleur match et créer la **bridge finale**

In [12]:
# 1 ligne par LCPInspireId : candidat avec le match_score le plus élevé
bridge_best = bridge_candidates.groupby("LCPInspireId", as_index=False).first()

# top1_minus_top2 : écart de match_score entre le 1er et le 2e candidat → indique l'ambiguïté
top2 = (
    bridge_candidates.groupby("LCPInspireId")
    .head(2)
    .sort_values(["LCPInspireId", "match_score"], ascending=[True, False])
)

score_gap_rows = []
for lcp_id, grp in top2.groupby("LCPInspireId"):
    scores = grp["match_score"].tolist()
    top1_minus_top2 = scores[0] - scores[1] if len(scores) > 1 else np.nan
    score_gap_rows.append((lcp_id, top1_minus_top2))

score_gap = pd.DataFrame(score_gap_rows, columns=["LCPInspireId", "top1_minus_top2"])
bridge_best = bridge_best.merge(score_gap, on="LCPInspireId", how="left")

# confidence_level : "faible" (match_score < 60), "moyenne" (60–80), "forte" (> 80)
bridge_best["confidence_level"] = pd.cut(
    bridge_best["match_score"],
    bins=[-1, 60, 80, 100],
    labels=["faible", "moyenne", "forte"]
)

# manual_review_required = True si confidence_level ≠ "forte" OU top1_minus_top2 < 5
bridge_best["manual_review_required"] = (
    (bridge_best["confidence_level"] != "forte")
    | (bridge_best["top1_minus_top2"].fillna(999) < 5)
)

bridge_best = bridge_best.sort_values("match_score", ascending=False).reset_index(drop=True)

display(bridge_best.head(10))


,LCPInspireId,InstallationInspireId,parent_facilityInspireId,lcp_name,installation_name,lcp_city,installation_city,lcp_longitude,lcp_latitude,installation_longitude,installation_latitude,distance_km,name_similarity,same_city_flag,id_guess_match,same_city_search_used,candidate_rank,match_score,top1_minus_top2,confidence_level,manual_review_required
0,BE.WA/095010101.PART,BE.WA/095010100.INSTALLATION,BE.WA/095010000.FACILITY,Installations de combustion,Combustion,Wanze,Wanze,5.207238,50.528587,5.207238,50.528587,0.000000,100.0,True,False,True,3,95.00,15.65,forte,False
1,BE.WA/098010101.PART,BE.WA/098010100.INSTALLATION,BE.WA/098010000.FACILITY,Installation de combustion,Combustion,Fontenoy,Fontenoy,3.477903,50.574960,3.477903,50.574960,0.000000,100.0,True,False,True,2,95.00,15.88,forte,False
2,BE.WA/225010101.PART,BE.WA/225010100.INSTALLATION,BE.WA/225010000.FACILITY,TGV 720 MWth,TGV 720 Mwth,Goutroux,Goutroux,4.433236,50.409810,4.433236,50.409810,0.000000,100.0,True,False,True,1,95.00,78.63,forte,False
3,BE.WA/215010101.PART,BE.WA/215010100.INSTALLATION,BE.WA/215010000.FACILITY,Installation de cogénération,Cogénération,Vielsalm,Vielsalm,5.954014,50.293080,5.954014,50.293080,0.000000,100.0,True,False,True,3,95.00,19.24,forte,False
4,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000026.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001163.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000988.FACILITY,ELECTRABEL TURBOJET ZEDELGEM,Electrabel Zedelgem,Zedelgem,Zedelgem,3.155440,51.130760,3.155410,51.130780,0.003054,100.0,True,False,True,1,94.94,32.61,forte,False
5,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000025.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001152.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000986.FACILITY,ELECTRABEL TURBOJET NOORDSCHOTE,Electrabel Noordschote,Houthulst,Houthulst,2.827520,50.956190,2.827520,50.956150,0.004448,100.0,True,False,True,1,94.92,53.17,forte,False
6,BE.BRU/100010003.PART,BE.BRU/100010012.INSTALLATION,BE.BRU/100010011.FACILITY,TURBOJET VOLTA,TURBOJET VOLTA,Bruxelles,Bruxelles,4.395500,50.813900,4.395460,50.813850,0.006230,100.0,True,False,True,1,94.89,72.16,forte,False
7,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000018.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001981.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000001847.FACILITY,ELECTRABEL - TJ BEERSE,Electrabel Beerse,Beerse,Beerse,4.852770,51.326420,4.852740,51.326370,0.005938,100.0,True,False,True,1,94.89,49.56,forte,False
8,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000027.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000791.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000637.FACILITY,ELECTRABEL TURBOJET ZELZATE,Electrabel turbojet Zelzate,Zelzate,Zelzate,3.794830,51.188340,3.794940,51.188370,0.008361,100.0,True,False,True,1,94.85,33.53,forte,False
9,BE.WA/103010101.PART,BE.WA/103010100.INSTALLATION,BE.WA/103010000.FACILITY,Installation de combustion (Turbine à contrepression) ERREUR DE DOUBLON,Combustion,VIRTON,Virton,5.504200,49.549800,5.504244,49.549896,0.011143,100.0,True,False,True,2,94.80,15.75,forte,False


## 8. Diagnostic qualité du bridge

In [13]:
print("Répartition des niveaux de confiance :")
display(bridge_best["confidence_level"].value_counts(dropna=False).to_frame("count"))

print("\nRépartition des cas à revoir manuellement :")
display(bridge_best["manual_review_required"].value_counts(dropna=False).to_frame("count"))

print("\nDistance moyenne du meilleur match :")
display(bridge_best["distance_km"].describe().to_frame("distance_km"))


Répartition des niveaux de confiance :


,count
confidence_level,
forte,86
moyenne,28
faible,0



Répartition des cas à revoir manuellement :


,count
manual_review_required,
False,64
True,50



Distance moyenne du meilleur match :


,distance_km
count,114.000000
mean,0.164528
std,0.200243
min,0.000000
25%,0.006011
50%,0.105089
75%,0.236375
max,1.098986


## 9. Cas à relire manuellement

`manual_review_required = True` quand :
- `confidence_level` ≠ `"forte"` → `match_score` < 80
- `top1_minus_top2` < 5 → deux candidats quasi ex-aequo

Colonnes clés pour la revue : `lcp_name` ↔ `installation_name`, `lcp_city` ↔ `installation_city`, `distance_km`, `name_similarity`, `match_score`.

In [14]:
to_review = bridge_best[
    (bridge_best["manual_review_required"] == True)
].sort_values(["match_score", "distance_km"], ascending=[True, True])

display(
    to_review[
        [
            "LCPInspireId", "InstallationInspireId", "parent_facilityInspireId",
            "lcp_name", "installation_name",
            "lcp_city", "installation_city",
            "distance_km", "name_similarity", "match_score",
            "confidence_level", "top1_minus_top2"
        ]
    ].head(30)
)


,LCPInspireId,InstallationInspireId,parent_facilityInspireId,lcp_name,installation_name,lcp_city,installation_city,distance_km,name_similarity,match_score,confidence_level,top1_minus_top2
113,BE.WA/117010101.PART,BE.WA/117010100.INSTALLATION,BE.WA/117010000.FACILITY,TG4 ( 2 turbines à gaz 2*161),Combustion,NaN,Angleur,0.000000,17.647059,60.29,moyenne,35.66
112,BE.WA/047010101.PART,BE.WA/047010200.INSTALLATION,BE.WA/047010000.FACILITY,Installation de cogénération,"Fabrication de matières plastiques de base (polymères PP, PE et PS)",Feluy,Feluy,0.797941,42.696629,63.18,moyenne,1.76
111,BE.EEA/BE0105.PART,BE.WA/103010200.INSTALLATION,BE.WA/103010000.FACILITY,Cellardennes 60m,Exploitation d'un CET de classe 5.1,VIRTON,Virton,0.398095,35.294118,68.29,moyenne,2.41
109,BE.WA/000002514.PART,BE.WA/119010100.INSTALLATION,BE.WA/119010000.FACILITY,TGVS2,Combustion,Seraing,Seraing,0.000000,13.333333,69.00,moyenne,46.92
110,BE.WA/119010101.PART,BE.WA/119010100.INSTALLATION,BE.WA/119010000.FACILITY,TGVS1,Combustion,Seraing,Seraing,0.000000,13.333333,69.00,moyenne,46.92
108,BE.EEA/BE0015.PART,BE.WA/113010100.INSTALLATION,BE.WA/113010000.FACILITY,Centrale Elec. Electrabel Awirs t5,Combustion,FLEMALLE-HAUTE,Flémalle-Haute,0.000780,13.953488,69.17,moyenne,47.35
106,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000048.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000629.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000503.FACILITY,IPPAS INVEST_LCP 1,SAPPI Lanaken,Lanaken,Lanaken,0.263900,32.258065,69.84,moyenne,10.03
107,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000049.PART,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000629.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000503.FACILITY,IPPAS INVEST_LCP 2,SAPPI Lanaken,Lanaken,Lanaken,0.263900,32.258065,69.84,moyenne,10.03
105,BE.WA/120010101.PART,BE.WA/120010100.INSTALLATION,BE.WA/120010000.FACILITY,Centrale TGV,Combustion,LIEGE,LIEGE,0.000000,18.181818,70.45,moyenne,0.51
104,BE.WA/114010101.PART,BE.WA/114010100.INSTALLATION,BE.WA/114010000.FACILITY,TURBINNE A GAZ (Saint-Ghislain I),Combustion,Baudour,Baudour,0.000000,19.512195,70.85,moyenne,49.55


## 10. Sauvegarde

| Variable | Fichier | Grain |
|---|---|---|
| `bridge_best` | `bridge_f5_f6_belgium.csv` | 1 ligne par `LCPInspireId` |
| `bridge_candidates` | `bridge_f5_f6_belgium_candidates.csv` | `LCPInspireId × candidat` |

In [15]:
output_dir = Path(".")
bridge_path = output_dir / f"bridge_f5_f6_{TARGET_COUNTRY.lower()}.csv"
candidates_path = output_dir / f"bridge_f5_f6_{TARGET_COUNTRY.lower()}_candidates.csv"

bridge_best.to_csv(bridge_path, index=False)
bridge_candidates.to_csv(candidates_path, index=False)

print("Bridge final sauvegardé :", bridge_path.resolve())
print("Candidats sauvegardés :", candidates_path.resolve())


Bridge final sauvegardé : /home/olivierpi/technofutur/15-final_project/belgian-environement-monitor/notebooks/bridge_f5_f6_belgium.csv
Candidats sauvegardés : /home/olivierpi/technofutur/15-final_project/belgian-environement-monitor/notebooks/bridge_f5_f6_belgium_candidates.csv


## 11. Enrichissement de F5_2

`f5.merge(bridge_best, on="LCPInspireId", how="left")` ajoute à chaque ligne de F5 :

| Colonne ajoutée | Source |
|---|---|
| `InstallationInspireId` | `bridge_best` |
| `parent_facilityInspireId` | `bridge_best` |
| `match_score` | `bridge_best` |
| `confidence_level` | `bridge_best` |
| `manual_review_required` | `bridge_best` |

**Grain de `f5_enriched`** : toujours `LCPInspireId × reportingYear × featureType` — jointure many-to-one, pas de duplication.

Permet ensuite de remonter : `F5_2` → `F6_1` → facilities → `F1_4_Air_Releases_Facilities`.

In [16]:
f5_enriched = f5.merge(
    bridge_best[
        [
            "LCPInspireId", "InstallationInspireId", "parent_facilityInspireId",
            "distance_km", "name_similarity", "match_score",
            "confidence_level", "manual_review_required"
        ]
    ],
    on="LCPInspireId",
    how="left"
)

display(f5_enriched.head(10))

f5_enriched_path = output_dir / f"f5_enriched_with_bridge_{TARGET_COUNTRY.lower()}.csv"
f5_enriched.to_csv(f5_enriched_path, index=False)

print("F5 enrichie sauvegardée :", f5_enriched_path.resolve())


,countryName,reportingYear,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,featureType,unit,featureValue,InstallationInspireId,parent_facilityInspireId,distance_km,name_similarity,match_score,confidence_level,manual_review_required
0,Belgium,2018,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000069.PART,VPK PAPER,Dendermonde,4.068840,51.015210,LCPCharacteristics,MW,65.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001715.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000687.FACILITY,0.041357,100.000000,94.24,forte,True
1,Belgium,2016,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000060.PART,TOTALENERGIES REFINERY ANTWERP_LCP 2,Antwerpen,4.326460,51.266090,LCPCharacteristics,MW,259.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000188.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000169.FACILITY,0.368864,100.000000,88.24,forte,False
2,Belgium,2016,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000067.PART,TURBO-JET ZEEBRUGGE,Brugge,3.192240,51.318090,LCPCharacteristics,MW,80.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001689.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000001589.FACILITY,0.026988,34.146341,74.75,moyenne,True
3,Belgium,2021,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000054.PART,TEREOS STARCH_SWEETENERS BELGIUM_LCP 1,Aalst,4.043870,50.937800,LCPCharacteristics,MW,163.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000966.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000717.FACILITY,0.049819,100.000000,94.09,forte,False
4,Belgium,2020,BE.WA/095010101.PART,Installations de combustion,Wanze,5.207238,50.528587,LCPCharacteristics,MW,125.0,BE.WA/095010100.INSTALLATION,BE.WA/095010000.FACILITY,0.000000,100.000000,95.00,forte,False
5,Belgium,2018,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000036.PART,FLUXYS-LNG-TERMINAL,Brugge,3.221190,51.351160,LCPCharacteristics,MW,105.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001189.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000997.FACILITY,0.269394,100.000000,90.06,forte,False
6,Belgium,2020,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000021.PART,ELECTRABEL CENTRALE KNIPPEGROEN,Gent,3.807130,51.158910,LCPCharacteristics,MW,750.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000000794.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000640.FACILITY,0.099312,89.795918,90.12,forte,False
7,Belgium,2024,BE.WA/225010101.PART,TGV 720 MWth,Goutroux,4.433236,50.409810,LCPCharacteristics,MW,720.0,BE.WA/225010100.INSTALLATION,BE.WA/225010000.FACILITY,0.000000,100.000000,95.00,forte,False
8,Belgium,2018,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000013.PART,COVESTRO_LCP2,Antwerpen,4.322620,51.296740,LCPCharacteristics,MW,78.0,https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallation/BE.VL.000001766.INSTALLATION,https://data.ied_registry.omgeving.vlaanderen.be/id/productionfacility/BE.VL.000000090.FACILITY,0.003476,76.190476,87.79,forte,False
9,Belgium,2020,BE.WA/049010701.PART,Solvay BUSG1,Jemeppe-Sur-Sambre,4.662693,50.447224,LCPCharacteristics,MW,58.8,BE.WA/000002135.INSTALLATION,BE.WA/049010000.FACILITY,0.000000,22.857143,71.86,moyenne,True


F5 enrichie sauvegardée : /home/olivierpi/technofutur/15-final_project/belgian-environement-monitor/notebooks/f5_enriched_with_bridge_belgium.csv


## Combien de lignes sont perdues au total ?

Avec le **bridge Belgique** généré, tu ne perds **aucune ligne** si tu fais l’enrichissement de `F5_2` par `LCPInspireId`.

### Résultat global

- **F5_2 Belgique** : **10 521 lignes**
- **LCP uniques** : **114**
- **Bridge final** : **114 correspondances**
- **LCP non appariés** : **0**
- **Lignes perdues après merge** : **0 / 10 521**, soit **0,0 %**

Autrement dit :

- au niveau des **LCP uniques**, tu ne perds rien ;
- au niveau des **lignes de F5_2 Belgique**, tu ne perds rien non plus.

---

## Nuance importante

Même si tu ne perds **aucune ligne**, cela ne veut pas dire que tout est **100 % validé**.

Dans le bridge :

- **86 matches** sont en confiance **forte**
- **28 matches** sont en confiance **moyenne**
- **50 LCP** sont marqués **à revoir manuellement**

---

## Deux façons d’interpréter le résultat

### 1. Si tu acceptes tout le bridge

- **Perte = 0 ligne**

### 2. Si tu gardes seulement les cas non marqués en revue manuelle

Tu conserves :

- **6 210 lignes**

Tu mets de côté :

- **4 311 lignes**

Soit :

- **40,98 %** des lignes de `F5_2 Belgique`

---

## Conclusion

Il y a donc **deux réponses possibles** :

### Cas 1 — Avec tout le bridge
**Tu ne perds aucune ligne.**

### Cas 2 — Avec une version stricte, sans les cas à revoir
**Tu perds 4 311 lignes sur 10 521**, soit environ **41 %**.

---

## Formulation conseillée dans le projet

> Le bridge couvre **100 % des LCP belges**, donc **aucune ligne n’est perdue** lors de l’enrichissement.  
> En revanche, une partie des correspondances reste de **confiance moyenne** et nécessite une **validation manuelle**.

## 12. Utilisation du bridge en aval

### Tables Silver recommandées

| Table | Grain |
|---|---|
| `silver_f5_lcp_energy` | `LCPInspireId × reportingYear` |
| `silver_bridge_lcp_installation` | `LCPInspireId` (1 ligne) |
| `silver_f6_installations` | `InstallationInspireId × reportingYear` |

### Chaîne de jointure

```
F5_2  (agrégé LCPInspireId × reportingYear)
  → bridge_best           (via LCPInspireId)
  → F6_1                  (via InstallationInspireId + reportingYear)
  → F1_4                  (via parent_facilityInspireId)
```

> Ce matching est **probabiliste**. Toujours exposer `match_score`, `confidence_level` et `manual_review_required` dans les analyses en aval.

## 13. Résumé

| Problème | Solution dans ce notebook |
|---|---|
| Grains différents | `groupby().agg()` avant matching |
| Pas de clé commune | Bridge : distance + `name_similarity` + `city_norm` + pattern ID |
| Fiabilité variable | `match_score` + `confidence_level` + `manual_review_required` |
| IDs hétérogènes | `LCPInspireId` (`.PART`) ↔ `InstallationInspireId` (`.INSTALLATION`) |

In [19]:
import sys
from pathlib import Path

# Ajoute la racine du projet au path Python
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
from src.utils.db import connect_to_db,load_table

engine = connect_to_db()

parameters of DATABASE sucessfuly loaded 
connection sucessful


In [28]:
df_load = pd.read_csv("bridge_f5_f6_belgium.csv")
df_load = df_load.rename(str.lower, axis='columns')
print(df_load.columns.tolist())

['lcpinspireid', 'installationinspireid', 'parent_facilityinspireid', 'lcp_name', 'installation_name', 'lcp_city', 'installation_city', 'lcp_longitude', 'lcp_latitude', 'installation_longitude', 'installation_latitude', 'distance_km', 'name_similarity', 'same_city_flag', 'id_guess_match', 'same_city_search_used', 'candidate_rank', 'match_score', 'top1_minus_top2', 'confidence_level', 'manual_review_required']


In [30]:

load_table(df_load,"bridge_lcp_installation","silver",engine)


load of silver.bridge_lcp_installation done
